## Preparing Dataset for Model Training

In [1]:
import pandas as pd
import json
import torch
from onehotencoding import OneHotEncoder


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/dhruv.patravali/miniconda3/envs/02601_project_env/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/dhruv.patravali/miniconda3/envs/02601_project_env/lib/python3.11/site-packages/traitlets/config/application.py", line 1080, in launch_instance
    app.start()
  File "/Users/dhruv.patravali/miniconda3/envs/02601_project_env/lib/python3.11/

In [2]:
# Load Datasets
train_data_path = '../data/clean/deeploc_train.csv'
val_data_path = '../data/clean/deeploc_val.csv'
test_data_path = '../data/clean/deeploc_test.csv'
aa_idx_path = '../configs/aa_to_idx.json'

train_df = pd.read_csv(train_data_path)
val_df = pd.read_csv(val_data_path)
test_df = pd.read_csv(test_data_path)
with open(aa_idx_path) as file:
    aa_to_idx = json.load(file)


In [3]:
train_df.head()

,protein_id,sequence,subcellular_location,membrane_status,train_test,num_unusual_aa,seq_length,label
0,Q9CQ84,MFLTVKLLLGRRCSLKVSGKESVATLKKLVSQHLQVPEEQQHLLFR...,Cytoplasm,S,train,0,188,1
1,Q64279,MNLVGSYAHHHHHHHSHPPHPMLHEPFLFGPASRCHQERPYFQSWL...,Nucleus,S,train,0,216,7
2,Q8INV7,MSNTRTVHSSTSISKMNSALQISCLLVVLGCLLGSGHCQSEAEFAA...,Extracellular,S,train,0,134,3
3,P10986,MCDDEVAALVVDNGSGMCKAGFAGDDAPRAVFPSIVGRPRHQGVMV...,Cytoplasm,S,train,0,376,1
4,Q9HA38,MILLQHAVLPPPKQPSPSPPMSVATRSTGTLQLPPQKPFGQEASLP...,Nucleus,U,train,0,289,7


In [4]:
# Author: Dhruv

# Sequence to Tensor Encoding
def encode_sequences(seq, aa_to_idx):
    token_ids = [aa_to_idx[aa] for aa in seq]
    return torch.tensor(token_ids, dtype=torch.long)

# Encode all Sequencecs in a dataframe
def encode_split(df, aa_to_idx):
    tensors = []
    for seq, label in zip(df['sequence'], df['label']):
        tensor = encode_sequences(seq, aa_to_idx)
        tensors.append((tensor, label))
    return tensors

# Encode our 3 df splits
train_data = encode_split(train_df, aa_to_idx)
val_data = encode_split(val_df, aa_to_idx)
test_data = encode_split(test_df, aa_to_idx)

In [5]:
# Author: Dhruv

# Workflow: prepare batches by length --> pad batch by padding each tensor

def seq_length(tensor_label_pair):
    return len(tensor_label_pair[0])

def prepare_batches(train_data):
    train_batches = []
    BATCH_SIZE = 32
    sorted_train = sorted(train_data, key=seq_length)
    for i in range(0, len(sorted_train), BATCH_SIZE):
        train_batches.append(sorted_train[i:i+BATCH_SIZE])
    return train_batches

def pad_tensor(tensor, target_length, aa_to_idx):
    padded_tensor = tensor.tolist()
    
    if target_length < len(tensor):
        raise ValueError("target length shorter than seq length")
    
    pad_length = target_length - len(tensor)

    for _ in range(pad_length):
        padded_tensor.append(aa_to_idx['PAD'])

    return torch.tensor(padded_tensor, dtype=torch.long)

def pad_batch(train_batch, aa_to_idx):
    padded_batch = []
    target_length = seq_length(train_batch[-1]) # length of longest tensor
    for tensor, label in train_batch:
        padded_tensor = pad_tensor(tensor, target_length, aa_to_idx)
        padded_batch.append((padded_tensor, label))
    return padded_batch

In [6]:
# Author: AI (generated test)

train_batches = prepare_batches(train_data)
padded_batch = pad_batch(train_batches[0], aa_to_idx)

sequences, labels = zip(*padded_batch)
inputs = torch.stack(sequences)
targets = torch.tensor(labels, dtype=torch.long)

print(inputs.shape)
print(targets.shape)

torch.Size([32, 48])
torch.Size([32])


In [8]:
one_hot_encoder = OneHotEncoder(aa_to_idx)
one_hot_inputs = one_hot_encoder.encode_batch(inputs)

In [9]:
one_hot_inputs.shape

torch.Size([32, 48, 22])